# チュートリアル 09: Human-in-the-Loop & インタラクティブワークフロー

##  学習目標
このノートブックを終えると、以下ができるようになります:
- 人間のレビューのためにワークフローを一時停止する**承認ゲート**を実装する
- ユーザー入力を要求する**インタラクティブエグゼキューター**を作成する
- 人間のフィードバックループを持つ**マルチターン会話**を構築する
- **コンテキスト変数**を使用して承認状態を追跡する
- 却下されたリクエストに対する**却下ハンドラー**を実装する
- **人間の監督が重要な場合とオプションの場合**を理解する

##  前提条件

このチュートリアルを始める前に、以下を完了しておく必要があります:
- **チュートリアル 05: マルチエージェントワークフロー** (Sequential、Concurrentパターン)
- **チュートリアル 07: 高度なワークフロー** (WorkflowBuilder、CustomExecutor)

##  Human-in-the-Loopとは？

**Human-in-the-Loop (HITL)** ワークフローは、人間の入力を得るために実行を一時停止します:
-  **承認ゲート** - 人間がAIの決定をレビューして承認/却下する
-  **インタラクティブフィードバック** - 人間が実行をガイドする入力を提供する
-  **品質管理** - 人間が重要な出力を検証する
-  **ポリシー準拠** - 人間がルールへの遵守を確保する

### HITLを使用する理由

| シナリオ | HITLを使う理由 |
|----------|----------||
| 金融取引 | 承認の法的要件 |
| コンテンツ公開 | 品質とブランドの安全性 |
| データ削除 | 取り消し不可能なアクション |
| 顧客コミュニケーション | 評判リスク |
| ポリシー違反 | 手動判断が必要 |
| 高価値の決定 | コスト/リスクの軽減 |

### このチュートリアルのHITLパターン

1. **承認ゲートパターン** - シンプルな承認/却下の決定
2. **インタラクティブリファインメント** - フィードバックを伴うマルチターン会話
3. **条件付き承認** - 決定に基づく異なるパス
4. **予算ベースの承認** - しきい値以下は自動承認、それ以外は人間が必要

---

## ステップ 1: セットアップとインポート

In [ ]:
import asyncio
from typing import cast


from agent_framework import (
    # Workflow builders
    WorkflowBuilder,
    # Core components
    Executor,
    AgentExecutor,
    AgentExecutorResponse,
    handler,
    WorkflowContext,
    # Events
    WorkflowEvent,
    AgentResponseUpdate,
    # Message types
    Message,
)
from agent_framework.orchestrations import SequentialBuilder
from agent_framework.azure import AzureAIAgentClient
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()
print(" Imports successful!")
print("🎭 Ready to build Human-in-the-Loop workflows!")

In [ ]:
import os
mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

In [ ]:
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import DefaultAzureCredential
from azure.identity.aio import AzureCliCredential, DefaultAzureCredential

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

def create_azure_openai_chat_client():
    """Azure OpenAI クライアントの作成"""
    return AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )


In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    ワークフローのグラフ情報を出力し、SVGで保存し、プレビューする関数
    
    Args:
        workflow: 可視化するワークフローオブジェクト
        filename: 保存するファイル名（拡張子なし）
        show_preview: プレビューを表示するかどうか
    
    Returns:
        保存されたSVGファイルのパス
    """
    # WorkflowVizオブジェクトを作成
    viz = WorkflowViz(workflow)
    
    # 3. SVGファイルとして保存
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"✅ SVGファイルを保存しました: {svg_path}")
        print("=" * 60)
        print()
        
        # 4. SVGをプレビュー表示
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("❌ エラー: 'graphviz'パッケージがインストールされていません")
        print("インストール方法: pip install agent-framework[viz] --pre")
        print(f"詳細: {e}")
        return None
    except Exception as e:
        print(f"❌ エラーが発生しました: {e}")
        return None

## ステップ 2: パターン 1 - シンプルな承認ゲート

最もシンプルなHITLパターン: AIが何かを生成し、人間が承認または却下します。

### フロー
```
ユーザーリクエスト → AIエージェント → 人間のレビュー → (承認) → 実行
                                       → (却下) → 停止
```

### 主要な概念
- **同期input()** - ユーザー入力を得るためにワークフローを一時停止
- **条件付きロジック** - 承認決定に基づいてルーティング
- **コンテキスト追跡** - ダウンストリームで使用するために承認状態を保存

In [ ]:
async def simple_approval_gate_demo():
    """
    シンプルな承認ゲートパターンをデモンストレーション:
    AIがメールを下書き → 人間が承認/却下 → 承認されたら送信
    """
    print("="*70)
    print("パターン 1: シンプルな承認ゲート")
    print("="*70)
    print("\nフロー: メール下書き → 人間のレビュー → 承認されたら送信\n")
    
    # メール下書きエージェントを作成
    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )
    email_agent = chat_client.as_agent(
        instructions="""
        あなたはプロフェッショナルなメールライターです。
        ユーザーのリクエストに基づいて、明確で簡潔、プロフェッショナルなメールを下書きしてください。
        メールは簡潔に（最大3〜4文）。
        必ず件名を含めてください。
        """,
        name="EmailDrafter",
    )
    
    # 人間の承認のための CustomExecutor
    class ApprovalGate(Executor):
        """人間の承認/却下のためにワークフローを一時停止"""
        
        def __init__(self):
            super().__init__(id="approval_gate")
        
        @handler
        async def review(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorResponse, str]) -> None:
            """
            下書きを人間に表示し、承認決定を取得。
            """
            # AgentExecutor は AgentExecutorResponse を返すため、それに合わせて取り出す
            draft = "No content"
            if response and getattr(response, "agent_run_response", None) and getattr(response.agent_run_response, "text", None):
                draft = response.agent_run_response.text
            elif response and getattr(response, "agent_response", None) and getattr(response.agent_response, "text", None):
                draft = response.agent_response.text
            
            print("\n" + "="*70)
            print("📧 レビュー用のメール下書き")
            print("="*70)
            print(draft)
            print("="*70)
            
            # 人間の入力のためにワークフローを一時停止
            while True:
                decision = input("\n👤 このメールを承認しますか？ (yes/no): ").strip().lower()
                
                if decision in ['yes', 'y']:
                    print(" メールが承認されました!")
                    await ctx.yield_output(f" 承認され送信されました\n\n{draft}")
                    break
                elif decision in ['no', 'n']:
                    print(" メールが却下されました!")
                    await ctx.yield_output(" 却下 - メールは送信されませんでした")
                    break
                else:
                    print("  'yes'または'no'を入力してください")
    
    # ワークフローを構築: エージェント → 承認ゲート
    email_executor = AgentExecutor(email_agent, id="EmailDrafter")
    approval_gate = ApprovalGate()
    
    workflow = (
        SequentialBuilder(participants=[email_executor, approval_gate])
        .build()
    )
    
    # 関数を使ってワークフローを可視化
    svg_path = visualize_workflow(
        workflow, 
        filename="simple_approval_gate_workflow.svg",
        show_preview=True
    )

    # テストリクエスト
    request = "来週金曜日午後2時に開催される四半期全社ミーティングをチームに発表するメールを作成してください。"
    
    print(f"\n リクエスト: {request}\n"
          )
    print("🤖 AIがメールを下書き中...\n")
    
    # ワークフローを実行
    final_output = None
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent) and event.type == "output":
            if not isinstance(event.data, AgentResponseUpdate):
                final_output = event.data
    
    # 最終結果を表示
    print("\n" + "="*70)
    print(" 最終結果")
    print("="*70)
    if final_output:
        print(final_output)
    print("="*70)
    
    print("\n 何が起こったか:")
    print("  1, AIがメールを下書き")
    print("  2, 人間のレビューのためにワークフローが一時停止")
    print("  3, 人間が承認または却下")
    print("  4, 決定に基づいてワークフローが完了")

await simple_approval_gate_demo()


このパターンでは **「`Executor` を使ったカスタム承認ゲートで、Sequential ワークフローの途中に人間の判断を挟めるか」** を確認しています。

具体的に検証しているポイント:

1. **`Executor` サブクラスによる HITL 実装** — `ApprovalGate(Executor)` で `@handler` を定義し、`input()` でワークフロー実行を同期的にブロックして人間の入力を待てること
2. **`AgentExecutor` → カスタム `Executor` の Sequential 連携** — `SequentialBuilder().participants([email_executor, approval_gate])` で、AI エージェントの出力（`AgentExecutorResponse`）がそのまま次の Executor のハンドラ引数として渡されること
3. **`ctx.yield_output()` による分岐出力** — 承認/却下の判断結果に応じて、ワークフローの最終出力を動的に切り替えられること
4. **前段の出力型の受け取り方** — `AgentExecutorResponse` から `agent_run_response.text` / `agent_response.text` を `getattr` で安全に取り出すパターン

**最もシンプルな HITL パターン（AI生成 → 人間が Yes/No → 結果確定）が `SequentialBuilder` + カスタム `Executor` で実現できる**ことの動作実証です。

## ステップ 3: パターン 2 - インタラクティブリファインメントループ

より洗練された: 人間がフィードバックを提供し、反復できるようにします。

### フロー
```
リクエスト → AI下書き → 人間のレビュー ┬→ 承認 → 完了
                                  └→ フィードバック → AI修正 → 人間のレビュー (ループ)
```

### 主要な概念
- **マルチターン会話** - 反復的な改善
- **フィードバック統合** - 人間がAIのリファインメントをガイド
- **ループ制御** - 終了条件 (承認または最大反復回数)

In [ ]:
async def interactive_refinement_demo():
    """
    インタラクティブリファインメントをデモンストレーション:
    AIが下書き → 人間がフィードバックを提供 → AIが修正 → 承認されるまで繰り返し
    """
    print("="*70)
    print("パターン 2: インタラクティブリファインメントループ")
    print("="*70)
    print("\nフロー: 下書き → レビュー → フィードバック → 修正 → 繰り返し\n")
    
    # コンテンツライターエージェントを作成
    # credential = AzureCliCredential()
    # chat_client = AzureAIAgentClient(async_credential=credential)
    
    # Azure OpenAI クライアントの作成
    # chat_client = AzureOpenAIChatClient(
    #     api_key=azure_openai_key,
    #     endpoint=azure_openai_endpoint,
    #     api_version=api_version,
    #     deployment_name=azure_deployment,
    # )

    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )

    writer_agent = chat_client.as_agent(
        instructions="""
        あなたはクリエイティブなブログ記事ライターです。
        リクエストされたトピックについて、魅力的で有益なブログ記事を書いてください。
        フィードバックを受けた場合は、それを反映して記事を修正してください。
        記事は簡潔に（3〜4段落）。
        """,
        name="BlogWriter",
    )
    
    # インタラクティブリファインメントエグゼキューター
    class InteractiveEditor(Executor):
        """人間がAIの出力を反復的にリファインできるようにする"""
        
        def __init__(self, agent, max_iterations=3):
            super().__init__(id="interactive_editor")
            self.agent = agent
            self.max_iterations = max_iterations
        
        @handler
        async def edit_loop(self, request: str, ctx: WorkflowContext[str, str]) -> None:
            """
            人間のフィードバックを伴うインタラクティブ編集ループ。
            """
            conversation_history = [Message(role="user", text=request)]
            iteration = 0
            
            while iteration < self.max_iterations:
                iteration += 1
                print(f"\n{'='*70}")
                print(f" 下書き {iteration} (最大 {self.max_iterations})")
                print("="*70)
                
                # AIの応答を取得
                response = await self.agent.run(conversation_history)
                draft = response.text
                
                print(draft)
                print("="*70)
                
                # 人間のフィードバックを取得
                decision = input("\n👤 (a)承認、(f)フィードバック、または(r)却下？ ").strip().lower()
                
                if decision in ['a', 'approve']:
                    print("\n コンテンツが承認されました!")
                    await ctx.yield_output(f" 承認 ({iteration}回の反復後)\n\n{draft}")
                    return
                
                elif decision in ['r', 'reject']:
                    print("\n コンテンツが却下されました!")
                    await ctx.yield_output(" 却下 - コンテンツは公開されませんでした")
                    return
                
                elif decision in ['f', 'feedback']:
                    feedback = input("\n どのような変更をご希望ですか？ ")
                    print(f"\n フィードバックが記録されました: {feedback}")
                    print(" AIがフィードバックに基づいて修正中...")
                    
                    # 会話履歴に追加
                    conversation_history.append(Message(role="assistant", text=draft))
                    conversation_history.append(
                        Message(
                            role="user", 
                            text=f"このフィードバックに基づいて修正してください: {feedback}"
                        )
                    )
                else:
                    print("無効なオプション。フィードバックリクエストとして扱います。")
                    feedback = input("\n どのような変更をご希望ですか？ ")
                    conversation_history.append(Message(role="assistant", text=draft))
                    conversation_history.append(
                        Message(
                            role="user", 
                            text=f"このフィードバックに基づいて修正してください: {feedback}"
                        )
                    )
            
            # 最大反復回数に達した
            print(f"\n  最大反復回数 ({self.max_iterations}) に達しました。")
            final_decision = input("👤 最終下書きを承認しますか？ (yes/no): ").strip().lower()
            
            if final_decision in ['yes', 'y']:
                await ctx.yield_output(f" 承認 (最大反復回数に達した)\n\n{draft}")
            else:
                await ctx.yield_output(" 却下 - 承認なしで最大反復回数に達しました")
    
    # ワークフローを構築
    editor = InteractiveEditor(writer_agent, max_iterations=3)
    
    workflow = (
        WorkflowBuilder(start_executor=editor)
        .build()
    )
    # 関数を使ってワークフローを可視化
    svg_path = visualize_workflow(
        workflow, 
        filename="interactive_refinement_workflow.svg",
        show_preview=True
    )

    # テストリクエスト
    request = "カスタマーサービスにおけるAIエージェントのメリットについてのブログ記事を書いてください。"
    
    print(f"\n リクエスト: {request}\n")
    print("🤖 AIが初期下書きを作成中...\n")
    
    # ワークフローを実行
    final_output = None
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent) and event.type == "output":
            if not isinstance(event.data, AgentResponseUpdate):
                final_output = event.data
    
    # 最終結果を表示
    print("\n" + "="*70)
    print(" 最終結果")
    print("="*70)
    if final_output:
        print(final_output)
    print("="*70)
    
    print("\n 何が起こったか:")
    print("  1, AIが初期下書きを作成")
    print("  2, 人間がレビューしてフィードバックを提供")
    print("  3, AIがフィードバックに基づいて修正")
    print("  4, 承認または最大反復回数まで続く")
    print("  5, マルチターン会話がコンテキストを維持")

await interactive_refinement_demo()

このパターンでは、**「単一の `Executor` 内部で、人間のフィードバックを受けてAIに再生成させるマルチターンの反復ループを実現できるか」** を確認しています。

具体的に検証しているポイント:

1. **Executor 内での自前ループによる HITL** — `InteractiveEditor(Executor)` の `@handler` 内に `while` ループを持ち、`input()` で 承認(a) / フィードバック(f) / 却下(r) の3択を取ることで、ワークフローエンジンに頼らず Executor 単体で反復制御できること
2. **会話履歴の手動管理によるマルチターン** — `conversation_history` リストに `Message(user/assistant)` を都度追加し、`agent.run(conversation_history)` に渡すことで、フィードバック内容を踏まえた修正版をAIに生成させられること（フレームワークの会話管理ではなく、自前で履歴を積む方式）
3. **`max_iterations` による安全な終了保証** — 人間がいつまでもフィードバックし続けても `max_iterations` で強制的にループを打ち切り、最終判断を求める設計
4. **`WorkflowBuilder().set_start_executor(editor).build()`** — Executor が1つだけの最小ワークフロー構成で、Sequential チェーンではなく単一ノードとして動作すること

パターン1（承認ゲート）との対比で言えば、**Yes/No の一回限りの判断ではなく、「フィードバック → AI修正 → 再レビュー」を繰り返せる反復改善ループが Executor 内のロジックだけで実装できる**ことの動作実証です。

## ステップ 4: パターン 3 - 分岐を伴う条件付き承認

承認決定に基づいて異なるワークフローパス。

### フロー
```
リクエスト → プランナーエージェント → 人間のレビュー ┬→ 承認 → 実行エージェント → 完了
                                       └→ 却下 → 代替パス → 完了
```

### 主要な概念
- **分岐ロジック** - 異なるダウンストリームエグゼキューター
- **コンテキスト受け渡し** - 承認状態を共有
- **フォールバックパス** - 却下を適切に処理

In [ ]:
async def conditional_approval_demo():
    """
    分岐を伴う条件付き承認をデモンストレーション:
    計画 → 承認/却下 → 決定に基づく異なるパス
    """
    print("="*70)
    print("パターン 3: 分岐を伴う条件付き承認")
    print("="*70)
    print("\nフロー: 計画 → レビュー → 分岐 (承認/却下パス)\n")
    
    # プランニングエージェントを作成
    # credential = AzureCliCredential()
    # chat_client = AzureAIAgentClient(async_credential=credential)
    
    # Azure OpenAI クライアントの作成
    # chat_client = AzureOpenAIChatClient(
    #     api_key=azure_openai_key,
    #     endpoint=azure_openai_endpoint,
    #     api_version=api_version,
    #     deployment_name=azure_deployment,
    # )

    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )
    planner_agent = chat_client.as_agent(
        instructions="""
        あなたはプロジェクト計画アシスタントです。
        タイムライン、予算、リソースを含む詳細なプロジェクト計画を作成してください。
        計画は簡潔に（4〜5の主要ポイント）。
        """,
        name="ProjectPlanner",
    )
    
    execution_agent = chat_client.as_agent(
        instructions="""
        あなたはプロジェクト実行アシスタントです。
        承認された計画に基づいて、段階的な実行ガイドを作成してください。
        具体的で実行可能なものにしてください。
        """,
        name="ExecutionPlanner",
    )
    
    alternative_agent = chat_client.as_agent(
        instructions="""
        あなたはクリエイティブな代替案アドバイザーです。
        元の計画が却下されたときに、代替アプローチを提案してください。
        検討すべき2〜3の異なるオプションを提供してください。
        """,
        name="AlternativesAdvisor",
    )
    
    # 分岐を伴う承認ゲート
    class BranchingApprovalGate(Executor):
        """決定に基づいて次のエグゼキューターを決定する承認ゲート"""
        
        def __init__(self):
            super().__init__(id="branching_approval")
            self.approved = False
        
        @handler
        async def review_and_branch(
            self,
            response: AgentExecutorResponse,
            ctx: WorkflowContext[AgentExecutorResponse, str],
        ) -> None:
            """
            計画をレビューして適切な次のステップにルーティング。
            """
            # AgentExecutorResponse からテキストを抽出（agent_run_response ではなく agent_response）
            plan = "No content"
            if response and getattr(response, "agent_response", None) and getattr(response.agent_response, "text", None):
                plan = response.agent_response.text
            
            print("\n" + "="*70)
            print("📋 レビュー用のプロジェクト計画")
            print("="*70)
            print(plan)
            print("="*70)
            
            # 承認決定を取得
            while True:
                decision = input("\n👤 この計画を承認しますか？ (yes/no): ").strip().lower()
                
                if decision in ['yes', 'y']:
                    print(" 計画が承認されました! 実行ガイドを作成中...")
                    self.approved = True
                    # ルーティング用プレフィックスを付けて送信（エッジ condition で分岐）
                    await ctx.send_message(
                        "APPROVED:\n" + f"次の計画の実行ステップを作成してください:\n\n{plan}"
                    )
                    break
                elif decision in ['no', 'n']:
                    print("計画が却下されました! 代替案を提案中...")
                    self.approved = False
                    await ctx.send_message(
                        "REJECTED:\n" + f"元の計画が却下されました。次の計画の代替案を提案してください:\n\n{plan}"
                    )
                    break
                else:
                    print("  'yes'または'no'を入力してください")
    
    # 結果ハンドラー
    class ResultHandler(Executor):
        """承認パスに基づいて最終出力をフォーマット"""
        
        def __init__(self, approval_gate):
            super().__init__(id="result_handler")
            self.approval_gate = approval_gate
        
        @handler
        async def format_result(
            self,
            response: AgentExecutorResponse,
            ctx: WorkflowContext[AgentExecutorResponse, str],
        ) -> None:
            # 応答からコンテンツを抽出
            content = "No content"
            if response and getattr(response, "agent_response", None) and getattr(response.agent_response, "text", None):
                content = response.agent_response.text
            
            if self.approval_gate.approved:
                output = f" 計画承認\n\n📋 実行ガイド:\n{content}"
            else:
                output = f" 計画却下\n\n 代替オプション:\n{content}"
            
            await ctx.yield_output(output)
    
    # 分岐を伴うワークフローを構築
    planner_executor = AgentExecutor(planner_agent, id="ProjectPlanner")
    approval_gate = BranchingApprovalGate()
    execution_executor = AgentExecutor(execution_agent, id="ExecutionPlanner")
    alternative_executor = AgentExecutor(alternative_agent, id="AlternativesAdvisor")
    result_handler = ResultHandler(approval_gate)
    
    # condition 付きエッジで承認/却下を分岐（文字列メッセージのプレフィックスで判定）
    workflow = (
        WorkflowBuilder(start_executor=planner_executor)
        .add_edge(planner_executor, approval_gate)
        .add_edge(
            approval_gate,
            execution_executor,
            condition=lambda msg: isinstance(msg, str) and msg.startswith("APPROVED:"),
        )
        .add_edge(
            approval_gate,
            alternative_executor,
            condition=lambda msg: isinstance(msg, str) and msg.startswith("REJECTED:"),
        )
        .add_edge(execution_executor, result_handler)
        .add_edge(alternative_executor, result_handler)
        .build()
    )
    # 関数を使ってワークフローを可視化
    svg_path = visualize_workflow(
        workflow,
        filename="conditional_approval_workflow.svg",
        show_preview=True
    )

    # テストリクエスト
    request = "3か月で新しいモバイルアプリをローンチする計画を作成してください。"
    
    print(f"\n リクエスト: {request}\n")
    print("🤖 AIがプロジェクト計画を作成中...\n")
    
    # ワークフローを実行
    final_output = None
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent) and event.type == "output":
            if not isinstance(event.data, AgentResponseUpdate):
                final_output = event.data
    
    # 最終結果を表示
    print("\n" + "="*70)
    print(" 最終結果")
    print("="*70)
    if final_output:
        print(final_output)
    print("="*70)
    
    print("\n 何が起こったか:")
    print("  1, AIが初期プロジェクト計画を作成")
    print("  2, 人間がレビューして承認/却下")
    print("  3, 決定に基づいてワークフローが分岐")
    print("  4, 承認 → 実行ガイドが作成")
    print("  5, 却下 → 代替オプションが提案")

await conditional_approval_demo()

このパターンでは、**「人間の承認/却下の判断結果に基づいて、`WorkflowBuilder` の条件付きエッジ（`add_edge` + `condition`）でワークフローを動的に分岐できるか」** を確認しています。

具体的に検証しているポイント:

1. **`ctx.send_message()` + `condition` ラムダによるルーティング** — 承認ゲートが `ctx.yield_output()` ではなく `ctx.send_message()` でメッセージを下流に送り、エッジの `condition=lambda msg: msg.startswith("APPROVED:")` / `"REJECTED:"` でどちらのパスに流すかを決定する仕組み
2. **DAG 型ワークフローの構築** — `SequentialBuilder`（直線）ではなく `WorkflowBuilder` + `add_edge` で「計画 → 承認ゲート → (承認なら)実行エージェント / (却下なら)代替エージェント → 結果ハンドラー」というダイヤモンド型の分岐・合流グラフを組めること
3. **Executor 間の状態共有** — `ResultHandler` が `self.approval_gate.approved` を参照して最終出力のフォーマットを切り替えており、Executor インスタンスのフィールドを使ったクロスノード状態共有が機能すること
4. **複数 `AgentExecutor` + 複数カスタム `Executor` の協調** — Planner / ExecutionPlanner / AlternativesAdvisor の3つのAIエージェントと、BranchingApprovalGate / ResultHandler の2つのカスタム Executor が1つのワークフローで連携すること

パターン1・2との対比で言えば、**HITL の判断が単なる出力の切り替えではなく、後続のワークフローパス自体を分岐させる（承認→実行計画、却下→代替案提案という別のAIエージェントに振り分ける）** ことの動作実証です。

## ステップ 5: パターン 4 - スマート承認 (自動承認 vs 手動)

低リスクのリクエストを自動的に承認し、高リスクのものには人間の承認が必要。

### フロー
```
リクエスト → リスク分析 ┬→ 低リスク → 自動承認 → 実行
                        └→ 高リスク → 人間のレビュー → (承認) → 実行
                                                     → (却下) → 停止
```

### 主要な概念
- **リスク評価** - ビジネスロジックが承認パスを決定
- **効率性** - 安全な場合は人間のレビューをスキップ
- **条件付きゲート** - 必要な場合のみ一時停止

In [ ]:
async def smart_approval_demo():
    """
    スマート承認をデモンストレーション:
    低価値の購入は自動承認、高価値には人間の承認が必要
    """
    print("="*70)
    print("パターン 4: スマート承認 (自動 vs 手動)")
    print("="*70)
    print("\nフロー: 分析 → ルート (自動承認 / 人間のレビュー)\n")
    
    # 購入推奨エージェント
    # credential = AzureCliCredential()
    # chat_client = AzureAIAgentClient(async_credential=credential)
    
    # Azure OpenAI クライアントの作成
    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )
    purchase_agent = chat_client.as_agent(
        instructions="""
        あなたは購入アシスタントです。
        リクエストに基づいて購入する具体的なアイテムを推奨してください。
        応答には推定コストを含めてください。
        形式: "アイテム: [名前], 推定コスト: $[金額], 理由: [理由]"
        """,
        name="PurchaseAdvisor",
    )
    
    # しきい値を持つスマート承認ゲート
    class SmartApprovalGate(Executor):
        """しきい値以下を自動承認、それ以上は人間の承認が必要"""
        
        def __init__(self, threshold=100):
            super().__init__(id="smart_approval")
            self.threshold = threshold
        
        @handler
        async def smart_approve(
            self,
            response: AgentExecutorResponse,
            ctx: WorkflowContext[AgentExecutorResponse, str],
        ) -> None:
            """
            コストを抽出し、自動承認またはヒューマンレビューをリクエスト。
            """
            # AgentExecutor は AgentExecutorResponse を返すため、それに合わせてテキストを抽出
            recommendation = "No content"
            if response and getattr(response, "agent_response", None) and getattr(response.agent_response, "text", None):
                recommendation = response.agent_response.text
            elif response and getattr(response, "full_conversation", None):
                conv = response.full_conversation or []
                if conv and getattr(conv[-1], "text", None):
                    recommendation = conv[-1].text
            
            print("\n" + "="*70)
            print(" 購入推奨")
            print("="*70)
            print(recommendation)
            print("="*70)
            
            # コストを抽出 (簡略版 - $amountを探す)
            import re
            cost_match = re.search(r'\$([0-9,]+)', recommendation or "")
            
            if cost_match:
                cost_str = cost_match.group(1).replace(',', '')
                cost = float(cost_str)
                
                print(f"\n 検出されたコスト: ${cost:.2f}")
                print(f"   承認しきい値: ${self.threshold:.2f}")
                
                if cost <= self.threshold:
                    print(f"\n 自動承認 (${self.threshold}しきい値以下)")
                    await ctx.yield_output(
                        f" 自動承認 (${cost:.2f} ≤ ${self.threshold:.2f})\n\n{recommendation}"
                    )
                else:
                    print(f"\n  人間の承認が必要 (${self.threshold}しきい値以上)")
                    
                    # 人間のレビューのために一時停止
                    while True:
                        decision = input("\n👤 この購入を承認しますか？ (yes/no): ").strip().lower()
                        
                        if decision in ['yes', 'y']:
                            print(" 人間によって承認されました!")
                            await ctx.yield_output(
                                f" 人間によって承認 (${cost:.2f} > ${self.threshold:.2f})\n\n{recommendation}"
                            )
                            break
                        elif decision in ['no', 'n']:
                            print(" 購入が却下されました!")
                            await ctx.yield_output(
                                f" 却下 (${cost:.2f} > ${self.threshold:.2f})\n\n理由: 承認者によって却下"
                            )
                            break
                        else:
                            print("  'yes'または'no'を入力してください")
            else:
                print("\n  コストを抽出できませんでした - 人間のレビューが必要")
                decision = input("\n👤 この購入を承認しますか？ (yes/no): ").strip().lower()
                
                if decision in ['yes', 'y']:
                    await ctx.yield_output(f" 承認\n\n{recommendation}")
                else:
                    await ctx.yield_output(" 却下")
    
    # ワークフローを構築
    purchase_executor = AgentExecutor(purchase_agent, id="PurchaseAdvisor")
    smart_gate = SmartApprovalGate(threshold=500.00)  # $500以下を自動承認
    
    workflow = (
        SequentialBuilder(participants=[purchase_executor, smart_gate])
        .build()
    )
    
    # 関数を使ってワークフローを可視化
    svg_path = visualize_workflow(
        workflow,
        filename="smart_approval_workflow.svg",
        show_preview=True
    )

    # 複数のシナリオでテスト
    test_cases = [
        "チーム用のオフィス用品（ペン、ノートなど）を推奨してください",
        "開発チームリーダー用の新しいノートパソコンを推奨してください",
    ]
    
    for i, request in enumerate(test_cases, 1):
        print("\n\n" + "#"*70)
        print(f"テストケース {i}")
        print("#"*70)
        print(f"\n リクエスト: {request}\n")
        print("🤖 AIがリクエストを分析中...\n")
        
        final_output = None
        async for event in workflow.run(request, stream=True):
            if isinstance(event, WorkflowEvent) and event.type == "output":
                if not isinstance(event.data, AgentResponseUpdate):
                    final_output = event.data
        
        print("\n" + "="*70)
        print(" 結果")
        print("="*70)
        if final_output:
            print(final_output)
        print("="*70)
    
    print("\n" + "="*70)
    print(" 何が起こったか:")
    print("="*70)
    print("  1, AIがコスト見積もり付きの購入を推奨")
    print("  2, スマートゲートがコストとしきい値を分析")
    print("  3, 低コストアイテムは即座に自動承認")
    print("  4, 高コストアイテムは人間のレビューが必要")
    print("  5, 効率性: 安全な場合はレビューをスキップ、リスクがある場合は必要")
    print("\n 本番環境のユースケース:")
    print("  • 金融取引 (< $Xを自動承認)")
    print("  • コンテンツモデレーション (低リスクを自動承認)")
    print("  • リソース割り当て (標準リクエストを自動承認)")
    print("  • データアクセス (公開は自動承認、機密はレビュー)")

await smart_approval_demo()

このパターンでは、**「AIの出力内容をビジネスルール（しきい値）でプログラム的に判定し、HITL を必要な場合のみ動的に発動できるか」** を確認しています。

具体的に検証しているポイント:

1. **条件付き HITL の選択的発動** — すべてのケースで人間に聞くのではなく、正規表現でコストを抽出し `cost <= threshold` なら `input()` を呼ばず自動承認、超過時のみ人間に判断を求める。HITL の「呼ぶ/呼ばない」をランタイムで切り替えられること
2. **AI出力のプログラム的パース** — `re.search(r'\$([0-9,]+)', ...)` でエージェントの自然言語応答から構造化データ（金額）を抽出し、ビジネスロジックの入力にする手法
3. **同一ワークフローでの複数テストケース実行** — 低コスト（オフィス用品）と高コスト（ノートパソコン）の2ケースを同じ `workflow` に流し、同一の承認ゲートが入力に応じて異なる挙動（自動承認 vs 手動承認）をとることの実証
4. **コストを抽出できない場合のフォールバック** — 正規表現がマッチしない場合は安全側に倒して人間のレビューを要求する防御的設計

パターン1〜3との対比で言えば、**HITL を「常に発動」ではなく「リスク/コストに応じて選択的に発動」するスマートゲートが、Executor 内の条件分岐だけで実現できる**ことの動作実証です。

##  重要なポイント

### 学んだこと

1. **承認ゲート**
   - ワークフローを一時停止するためにエグゼキューター内で`input()`を使用
   - 人間の承認/却下決定を取得
   - 決定に基づいてワークフローをルーティング

2. **インタラクティブリファインメント**
   - フィードバックを伴うマルチターン会話
   - 会話履歴を維持
   - 承認または最大反復回数まで反復

3. **条件付き分岐**
   - 承認に基づく異なるパス
   - エグゼキューター間で状態を共有
   - 却下を適切に処理

4. **スマート承認**
   - ビジネスロジックが承認パスを決定
   - 低リスクアイテムを自動承認
   - 高リスクには人間のレビューが必要
   - 効率性と監督のバランス

### HITLのベストプラクティス

**HITLを使用する場合:**
-  高リスクの決定 (金融、法的)
-  取り消し不可能なアクション (削除、公開)
-  コンプライアンス要件 (承認ポリシー)
-  品質管理 (ブランドの安全性)
-  AIが処理できないエッジケース

**HITLをスキップする場合:**
-  高ボリューム、低リスクのタスク
-  明確に定義された定型作業
-  リアルタイム要件
-  開発/テスト環境

### 本番環境パターン

```python
# シンプルな承認ゲート
class ApprovalGate(Executor):
    @handler
    async def review(self, content: str, ctx: WorkflowContext[str, str]):
        decision = input("承認しますか？ (yes/no): ")
        if decision == 'yes':
            await ctx.yield_output(f"承認: {content}")
        else:
            await ctx.yield_output("却下")

# しきい値を持つスマート承認
class SmartGate(Executor):
    def __init__(self, threshold):
        super().__init__(id="smart_gate")
        self.threshold = threshold
    
    @handler
    async def review(self, data: dict, ctx: WorkflowContext):
        if data['risk_score'] < self.threshold:
            await ctx.yield_output("自動承認")
        else:
            decision = input("高リスク - 承認しますか？ ")
            await ctx.yield_output(decision)
```

### 実世界のアプリケーション

1. **コンテンツ公開パイプライン**
   - AIがコンテンツを下書き → 編集者がレビュー → 承認されたら公開

2. **カスタマーサポートエスカレーション**
   - AIが定型問い合わせを処理 → 複雑なものを人間にエスカレート → 人間がガイダンスを提供

3. **金融承認ワークフロー**
   - < $1000を自動承認 → マネージャー承認 $1000-$10k → エグゼクティブ承認 > $10k

4. **データアクセス制御**
   - 公開データを自動承認 → 機密データには承認が必要

---

##  練習問題

### 演習 1: 多段階承認

段階的承認を伴うワークフローを作成:
```
リクエスト → マネージャー ($0-$5k) → ディレクター ($5k-$25k) → CFO (>$25k)
```

**ヒント:** 異なるしきい値を持つネストされた承認ゲートを使用。

### 演習 2: メモリを持つフィードバックループ

以下を行うインタラクティブシステムを作成:
- 以前のフィードバックを記憶
- 反復ごとに改善を表示
- ユーザーの好みを学習

**ヒント:** 会話履歴を維持して参照。

### 演習 3: 承認分析

以下を追跡してレポート:
- 自動承認率
- 手動承認率
- 平均レビュー時間
- 却下理由

**ヒント:** 承認ゲートにログを追加。

In [ ]:
# 演習プレイグラウンド - ここで解決策を実装してください!

async def multi_level_approval_exercise():
    """
    演習 1: 段階的承認ワークフローを構築。
    """
    # TODO: 多段階承認を実装
    pass

async def feedback_memory_exercise():
    """
    演習 2: メモリを持つフィードバックループを構築。
    """
    # TODO: コンテキストを持つフィードバックループを実装
    pass

async def approval_analytics_exercise():
    """
    演習 3: 承認メトリクスを追跡。
    """
    # TODO: 分析追跡を実装
    pass

print(" 演習テンプレート準備完了 - 上記で解決策を実装してください!")

##  次は？

おめでとうございます! Human-in-the-Loopパターンをマスターしました!

以下ができるようになりました:
-  人間の承認のためにワークフローを一時停止
-  インタラクティブなリファインメントループを作成
-  決定に基づく条件付き分岐を実装
-  しきい値を持つスマート承認ゲートを構築
-  人間の監督が必要な場合を選択

---

### クイックリファレンス

**承認ゲートパターン:**
```python
class ApprovalGate(Executor):
    def __init__(self):
        super().__init__(id="approval")
    
    @handler
    async def review(self, content: str, ctx: WorkflowContext[str, str]):
        print(f"レビュー: {content}")
        decision = input("承認しますか？ (yes/no): ")
        
        if decision == 'yes':
            await ctx.yield_output(f"承認: {content}")
        else:
            await ctx.yield_output("却下")
```

**インタラクティブループパターン:**
```python
conversation = [Message(role="user", text=request)]
for i in range(max_iterations):
    response = await agent.run(conversation)
    feedback = input("フィードバック: ")
    conversation.append(Message(role="assistant", text=response.text))
    conversation.append(Message(role="user", text=feedback))
```

**スマート承認パターン:**
```python
if risk_score < threshold:
    print("自動承認")
    await ctx.yield_output(content)
else:
    decision = input("高リスク - 承認しますか？ ")
    if decision == 'yes':
        await ctx.yield_output(content)
```

---

** チュートリアル08の完了、お疲れ様でした!**